# Diabetes Risk Screening Tool — End-to-End ML Pipeline

**Dataset:** CDC BRFSS 2015 *Diabetes Health Indicators* (Kaggle, `diabetes_binary_health_indicators_BRFSS2015.csv`, original imbalanced binary file).  
**Target:** `Diabetes_binary` — in this file `0` = no diabetes, `1` = **prediabetes or diabetes** (self-reported).

## Scientific limitations (read first)
* The data are **self-reported survey answers** from 2015 (BRFSS), not clinical measurements or lab tests.
* The label is a self-reported status at survey time. The model therefore finds **associations with reported status**; it does **not** predict *future* diabetes onset and is **not** a clinically validated diagnostic system.
* Class `1` merges prediabetes and diabetes; type 1 / type 2 are not distinguished.
* Survey weights, sampling design, and recall/response bias are ignored; the US adult population of 2015 may not represent other populations or today.
* Features are coarse (e.g. age in 13 bands, income in 8 bands) and many people share identical feature vectors, which limits achievable performance.
* Feature importance and SHAP values describe **model behaviour**, not causes.
* When class weights/SMOTE are used, model scores are **not calibrated probabilities**.

## Workflow
1. Setup & data loading → 2. EDA & data-quality checks → 3. Split & preprocessing → 4. Imbalance strategy comparison (CV) → 5. Hyper-parameter tuning (CV) → 6. Validation comparison → 7. Threshold selection (validation only) → 8. Single test evaluation → 9. Explainability → 10. Save artifacts & inference test → 11. Streamlit app.

## 1. Setup

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {"imblearn": "imbalanced-learn", "xgboost": "xgboost", "shap": "shap", "joblib": "joblib"}
OPTIONAL_PACKAGES = {"streamlit": "streamlit"}  # only needed to run the app

def ensure_packages(packages, required):
    missing = [pip_name for module, pip_name in packages.items() if importlib.util.find_spec(module) is None]
    if not missing:
        return
    print("Installing missing packages:", missing)
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    except subprocess.CalledProcessError:
        if required:
            raise
        print(f"Could not install optional packages {missing}; continuing (they are only needed to run the app).")

ensure_packages(REQUIRED_PACKAGES, required=True)
ensure_packages(OPTIONAL_PACKAGES, required=False)

In [ ]:
import json
import os
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import sklearn
import xgboost
import imblearn
from scipy.stats import loguniform, uniform
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, classification_report,
                             confusion_matrix, f1_score, fbeta_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*ill-defined.*")  # CV folds where a model predicts no positives at 0.5
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

# ---------------------------- Configuration ----------------------------
RANDOM_STATE = 42
TARGET = "Diabetes_binary"
TEST_SIZE = 0.15           # final hold-out set, touched exactly once
VALIDATION_SIZE = 0.15     # of the full data; used for model selection and threshold selection
TUNING_SAMPLE_SIZE = 50_000  # stratified subsample of TRAIN used for CV comparison/tuning (Kaggle CPU budget)
CV_FOLDS = 3
N_ITER = {"Logistic Regression": 8, "Decision Tree": 12, "Random Forest": 8, "XGBoost": 12}
F_BETA = 2.0               # screening: recall is weighted more than precision when picking the threshold
SIMPLICITY_TOLERANCE = 0.002  # PR-AUC difference treated as a tie between imbalance strategies

np.random.seed(RANDOM_STATE)

# Kaggle: only /kaggle/working is writable. Falls back to ./working when run elsewhere.
WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("working")
FIGURE_DIR = WORKING_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def show(fig, filename):
    # Save a figure under /kaggle/working/figures and display it.
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / filename, dpi=130, bbox_inches="tight")
    plt.show()

print("Working directory:", WORKING_DIR.resolve())
print({lib.__name__: lib.__version__ for lib in (sklearn, xgboost, imblearn, shap, pd, np)})

## 2. Load data (locating the CSV programmatically)

In [ ]:
CSV_NAME = "diabetes_binary_health_indicators_BRFSS2015.csv"
EXPECTED_PATH = Path("/kaggle/input/diabetes-health-indicators-dataset") / CSV_NAME

def locate_dataset():
    # Optional override for running outside Kaggle: set the DIABETES_CSV environment variable.
    override = os.environ.get("DIABETES_CSV")
    if override and Path(override).is_file():
        return Path(override)
    if EXPECTED_PATH.is_file():
        return EXPECTED_PATH
    print(f"Expected path not found: {EXPECTED_PATH}\nInspecting /kaggle/input ...")
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for path in sorted(input_root.rglob("*")):
            if path.is_file():
                print("  ", path)
        matches = sorted(input_root.rglob(CSV_NAME))
        if matches:
            return matches[0]
    raise FileNotFoundError(
        f"Could not find {CSV_NAME}. In the Kaggle notebook use 'Add Input' and attach the dataset "
        "'alexteboul/diabetes-health-indicators-dataset'."
    )

DATA_PATH = locate_dataset()
print("Using dataset:", DATA_PATH)
df = pd.read_csv(DATA_PATH)

## 3. Data understanding & EDA

In [ ]:
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
display(df.dtypes.to_frame("dtype").T)
display(df.head())
display(df.sample(5, random_state=RANDOM_STATE))

In [ ]:
# Feature groups, defined from the BRFSS codebook for this dataset.
CONTINUOUS_FEATURES = ["BMI", "MentHlth", "PhysHlth"]               # numeric measures / counts of days
ORDINAL_FEATURES = ["GenHlth", "Age", "Education", "Income"]         # ordered category codes
BINARY_FEATURES = ["HighBP", "HighChol", "CholCheck", "Smoker", "Stroke", "HeartDiseaseorAttack",
                   "PhysActivity", "Fruits", "Veggies", "HvyAlcoholConsump", "AnyHealthcare",
                   "NoDocbcCost", "DiffWalk", "Sex"]
FEATURES = [c for c in df.columns if c != TARGET]  # keep the original column order, target excluded

assert set(FEATURES) == set(CONTINUOUS_FEATURES + ORDINAL_FEATURES + BINARY_FEATURES), "Unexpected columns"
assert TARGET in df.columns and TARGET not in FEATURES
assert set(df[TARGET].unique()) <= {0, 1, 0.0, 1.0}, "Target is not binary 0/1"
assert all(set(df[c].unique()) <= {0, 1, 0.0, 1.0} for c in BINARY_FEATURES), "A 'binary' feature has other values"
print("Schema checks passed. 0 = no diabetes, 1 = prediabetes or diabetes (self-reported).")

In [ ]:
# Missing values, duplicates
missing = df.isna().sum()
print("Total missing values:", int(missing.sum()))
display(missing[missing > 0] if missing.sum() else pd.Series({"(no missing values)": 0}))

n_duplicates = int(df.duplicated().sum())
print(f"\nExact duplicate rows: {n_duplicates:,} ({n_duplicates / len(df):.1%} of rows)")

# Do identical feature vectors ever carry different labels? (irreducible label noise)
pattern_stats = df.groupby(FEATURES)[TARGET].agg(["size", "nunique"])
conflicting = pattern_stats[pattern_stats["nunique"] > 1]
print(f"Distinct feature patterns: {len(pattern_stats):,}")
print(f"Patterns with conflicting labels: {len(conflicting):,} covering {int(conflicting['size'].sum()):,} rows "
      f"({conflicting['size'].sum() / len(df):.1%} of all rows)")

**Duplicate handling decision.** The BRFSS file has no respondent ID and every feature is a coarse code, so identical rows are expected to be *different people who gave the same answers*. Removing them would distort prevalence and the class balance, so **duplicates are kept**. The consequence — some test rows may exactly match training rows, which can flatter test metrics slightly — is handled by a transparent diagnostic in the final evaluation (metrics on test rows that do *not* appear in the training set). Patterns with conflicting labels show a ceiling on achievable performance.

In [ ]:
# Target distribution / class imbalance
counts = df[TARGET].value_counts().sort_index()
share = df[TARGET].value_counts(normalize=True).sort_index()
print(pd.DataFrame({"count": counts, "share": share.round(4)}))
print(f"Imbalance ratio (negative : positive) = {counts[0] / counts[1]:.2f} : 1")

fig, ax = plt.subplots(figsize=(5, 3.5))
sns.countplot(x=TARGET, data=df, ax=ax)
for container in ax.containers:
    ax.bar_label(container, fmt="{:,.0f}")
ax.set_xticks([0, 1], ["0: no diabetes", "1: prediabetes/diabetes"])
ax.set_title("Target distribution")
show(fig, "01_target_distribution.png")

In [ ]:
display(df.describe().T.round(3))

# Investigating zeros instead of blindly treating them as missing
zero_report = pd.DataFrame({
    "n_zero": (df[FEATURES] == 0).sum(),
    "share_zero": (df[FEATURES] == 0).mean().round(3),
    "min": df[FEATURES].min(),
    "max": df[FEATURES].max(),
})
display(zero_report)
print("BMI quantiles:", df["BMI"].quantile([0, .001, .01, .5, .99, .999, 1]).round(1).to_dict())
print("Rows with BMI > 60:", int((df["BMI"] > 60).sum()), "| BMI < 15:", int((df["BMI"] < 15).sum()))

**Zero / outlier decisions.**
* Binary features: `0` means *No* — a valid answer, not a missing value.
* `MentHlth` / `PhysHlth`: `0` means *zero days of poor health* — valid and very common.
* `BMI` has no zeros (minimum is physiologically plausible). Extreme values are rare but real self-reported values, so they are **kept**; tree models are insensitive to them and the logistic regression uses standardisation. No imputation is needed on the training data, but imputers stay in the pipeline so inference is robust.

In [ ]:
# Continuous features: distributions by target
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, column in zip(axes, CONTINUOUS_FEATURES):
    sns.histplot(data=df, x=column, hue=TARGET, stat="density", common_norm=False, element="step",
                 bins=30, ax=ax, palette={0: "tab:blue", 1: "tab:red"})
    ax.set_title(f"{column} by target")
show(fig, "02_continuous_distributions.png")

# Binary features: share of positives per answer
binary_rows = []
for column in BINARY_FEATURES:
    rate = df.groupby(column)[TARGET].mean()
    binary_rows.append({"feature": column, "rate_if_0": rate.get(0, np.nan), "rate_if_1": rate.get(1, np.nan)})
binary_rates = pd.DataFrame(binary_rows).set_index("feature")
fig, ax = plt.subplots(figsize=(9, 6))
binary_rates.plot.barh(ax=ax)
ax.set_xlabel("Share with Diabetes_binary = 1")
ax.set_title("Positive rate by binary feature value")
show(fig, "03_binary_vs_target.png")

# Ordinal features: positive rate per level
fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))
for ax, column in zip(axes, ORDINAL_FEATURES):
    df.groupby(column)[TARGET].mean().plot(marker="o", ax=ax)
    ax.set_title(f"Positive rate by {column}")
    ax.set_ylabel("Share with Diabetes_binary = 1")
show(fig, "04_ordinal_vs_target.png")

In [ ]:
# Correlation heatmap (Spearman is rank-based and sensible for binary/ordinal codes)
correlation = df.corr(method="spearman")
fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(correlation, cmap="coolwarm", center=0, annot=True, fmt=".2f", annot_kws={"size": 6}, ax=ax)
ax.set_title("Spearman correlation matrix")
show(fig, "05_correlation_heatmap.png")

target_corr = correlation[TARGET].drop(TARGET).sort_values()
fig, ax = plt.subplots(figsize=(7, 6))
target_corr.plot.barh(ax=ax)
ax.set_title("Spearman correlation with target (association only, not causation)")
show(fig, "06_target_correlation.png")

feature_corr = correlation.loc[FEATURES, FEATURES].abs().where(~np.eye(len(FEATURES), dtype=bool))
print("Strongest feature-feature correlation:", round(float(feature_corr.max().max()), 3),
      "(highly redundant pairs, if any, would justify dropping a feature; trees/regularised LR tolerate moderate collinearity).")

### Data-quality summary
* Missing values were counted above; the pipeline includes imputers anyway so inference stays robust if a value is ever missing.
* Exact duplicates were counted above; they are retained (see decision above).
* Identical feature patterns can carry different labels (counted above) → irreducible noise; no model can be perfect.
* Ordinal codes (Age, Income, Education, GenHlth) are treated as ordered numerics; they are **not** one-hot encoded because their order is meaningful and one-hot would add many sparse columns.
* The dataset is imbalanced (see counts above) so accuracy alone is misleading.

## 4. Stratified train / validation / test split

In [ ]:
X = df[FEATURES].copy()
y = df[TARGET].astype(int).copy()

# 1) Hold out the test set first. It is not used for any decision until the very end.
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)

# 2) Split the rest into train / validation (validation = VALIDATION_SIZE of the full dataset).
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=VALIDATION_SIZE / (1 - TEST_SIZE),
    stratify=y_trainval, random_state=RANDOM_STATE)

split_summary = pd.DataFrame({
    "rows": [len(y_train), len(y_val), len(y_test)],
    "positive_rate": [y_train.mean(), y_val.mean(), y_test.mean()],
}, index=["train", "validation", "test"])
display(split_summary.round(4))

positive_weight = float((y_train == 0).sum() / (y_train == 1).sum())  # for XGBoost scale_pos_weight
print(f"Negative/positive ratio in TRAIN: {positive_weight:.2f}")

# Stratified subsample of TRAIN for cross-validated comparison/tuning (keeps Kaggle runtime reasonable).
X_tune, _, y_tune, _ = train_test_split(
    X_train, y_train, train_size=min(TUNING_SAMPLE_SIZE, len(X_train) - 1),
    stratify=y_train, random_state=RANDOM_STATE)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
print("Tuning subsample:", X_tune.shape, "| positive rate:", round(y_tune.mean(), 4))

## 5. Preprocessing and model pipelines

| Feature group | Handling |
|---|---|
| Continuous (`BMI`, `MentHlth`, `PhysHlth`) | median imputation; **StandardScaler only for Logistic Regression** (trees/XGBoost are scale-invariant) |
| Ordinal (`GenHlth`, `Age`, `Education`, `Income`) | most-frequent imputation; kept as ordered numbers; scaled only for Logistic Regression |
| Binary (14 features) | most-frequent imputation; already 0/1, never scaled |

Everything lives inside a `Pipeline`, so imputers/scalers are **fit on training folds only**. SMOTE is a pipeline step, so it is applied **only to the training portion of each CV fold / the training set** and never to validation or test data (imblearn skips samplers at `predict` time).

In [ ]:
def build_preprocessor(scale_numeric):
    continuous_steps = [("imputer", SimpleImputer(strategy="median"))]
    ordinal_steps = [("imputer", SimpleImputer(strategy="most_frequent"))]
    if scale_numeric:
        continuous_steps.append(("scaler", StandardScaler()))
        ordinal_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer(
        transformers=[
            ("continuous", Pipeline(continuous_steps), CONTINUOUS_FEATURES),
            ("ordinal", Pipeline(ordinal_steps), ORDINAL_FEATURES),
            ("binary", Pipeline([("imputer", SimpleImputer(strategy="most_frequent"))]), BINARY_FEATURES),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

MODEL_NAMES = ["Logistic Regression", "Decision Tree", "Random Forest", "XGBoost"]
STRATEGIES = ["none", "class_weight", "smote"]  # ordered from simplest to most complex

def make_estimator(model_name, strategy):
    class_weight = "balanced" if strategy == "class_weight" else None
    if model_name == "Logistic Regression":
        return LogisticRegression(max_iter=2000, class_weight=class_weight, random_state=RANDOM_STATE)
    if model_name == "Decision Tree":
        return DecisionTreeClassifier(max_depth=8, min_samples_leaf=50, class_weight=class_weight,
                                      random_state=RANDOM_STATE)
    if model_name == "Random Forest":
        return RandomForestClassifier(n_estimators=150, max_depth=12, min_samples_leaf=20,
                                      class_weight=class_weight, n_jobs=-1, random_state=RANDOM_STATE)
    if model_name == "XGBoost":
        return XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=4, subsample=0.8,
                             colsample_bytree=0.8, tree_method="hist", eval_metric="logloss",
                             scale_pos_weight=positive_weight if strategy == "class_weight" else 1.0,
                             n_jobs=-1, random_state=RANDOM_STATE)
    raise ValueError(model_name)

def make_pipeline(model_name, strategy):
    steps = [("preprocess", build_preprocessor(scale_numeric=(model_name == "Logistic Regression")))]
    if strategy == "smote":
        steps.append(("smote", SMOTE(random_state=RANDOM_STATE)))
    steps.append(("model", make_estimator(model_name, strategy)))
    return ImbPipeline(steps) if strategy == "smote" else Pipeline(steps)

## 6. Class-imbalance strategies: baseline vs class weights vs SMOTE
Each model × strategy is compared with stratified 3-fold CV on the training subsample. Precision/recall/F1 here use the default 0.5 cut-off, which **shifts with the strategy** (class weights/SMOTE move the operating point), so the **threshold-free PR-AUC** is the selection criterion, and the operating point is chosen later on validation data. If two strategies are within a small PR-AUC tolerance, the simplest one wins.

In [ ]:
SCORING = {"roc_auc": "roc_auc", "pr_auc": "average_precision", "recall": "recall",
           "precision": "precision", "f1": "f1"}

strategy_rows = []
for model_name in MODEL_NAMES:
    for strategy in STRATEGIES:
        started = time.time()
        scores = cross_validate(make_pipeline(model_name, strategy), X_tune, y_tune, cv=cv,
                                scoring=SCORING, n_jobs=1, error_score="raise")
        row = {"model": model_name, "strategy": strategy,
               **{metric: scores[f"test_{metric}"].mean() for metric in SCORING},
               "pr_auc_std": scores["test_pr_auc"].std(), "seconds": time.time() - started}
        strategy_rows.append(row)
        print(f"{model_name:20s} {strategy:13s} PR-AUC={row['pr_auc']:.4f}  ROC-AUC={row['roc_auc']:.4f}  "
              f"recall@0.5={row['recall']:.3f}  precision@0.5={row['precision']:.3f}  ({row['seconds']:.0f}s)")

strategy_table = pd.DataFrame(strategy_rows)
display(strategy_table.round(4))

def pick_strategy(model_name):
    subset = strategy_table[strategy_table["model"] == model_name].set_index("strategy")
    best_pr_auc = subset["pr_auc"].max()
    for strategy in STRATEGIES:  # simplest first
        if subset.loc[strategy, "pr_auc"] >= best_pr_auc - SIMPLICITY_TOLERANCE:
            return strategy

BEST_STRATEGY = {name: pick_strategy(name) for name in MODEL_NAMES}
print("\nSelected imbalance strategy per model:", BEST_STRATEGY)

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=strategy_table, x="model", y="pr_auc", hue="strategy", ax=ax)
ax.set_title("Cross-validated PR-AUC by imbalance strategy (training subsample)")
ax.set_ylim(strategy_table["pr_auc"].min() * 0.9, strategy_table["pr_auc"].max() * 1.05)
show(fig, "07_imbalance_strategy_comparison.png")

## 7. Hyper-parameter tuning (stratified CV, optimising PR-AUC)
`RandomizedSearchCV` runs on the training subsample with each model's selected imbalance strategy (resampling happens inside each CV training fold). The best parameters are then used to **refit on the full training set**. Search spaces are deliberately modest for Kaggle CPU limits.

In [ ]:
PARAM_DISTRIBUTIONS = {
    "Logistic Regression": {"model__C": loguniform(1e-3, 1e1)},
    "Decision Tree": {"model__max_depth": [4, 6, 8, 10, 12], "model__min_samples_leaf": [20, 50, 100, 200],
                      "model__criterion": ["gini", "entropy"]},
    "Random Forest": {"model__n_estimators": [100, 200, 300], "model__max_depth": [8, 12, 16],
                      "model__min_samples_leaf": [5, 10, 20, 50], "model__max_features": ["sqrt", "log2"]},
    "XGBoost": {"model__n_estimators": [200, 300, 500], "model__learning_rate": loguniform(0.02, 0.2),
                "model__max_depth": [3, 4, 5, 6], "model__subsample": uniform(0.7, 0.3),
                "model__colsample_bytree": uniform(0.6, 0.4), "model__min_child_weight": [1, 5, 10],
                "model__reg_lambda": loguniform(0.5, 10)},
}

tuned_models, tuning_rows = {}, []
for model_name in MODEL_NAMES:
    started = time.time()
    pipeline = make_pipeline(model_name, BEST_STRATEGY[model_name])
    search = RandomizedSearchCV(pipeline, PARAM_DISTRIBUTIONS[model_name], n_iter=N_ITER[model_name],
                                scoring="average_precision", cv=cv, random_state=RANDOM_STATE,
                                n_jobs=1, refit=False, error_score="raise")
    search.fit(X_tune, y_tune)
    best_params = search.best_params_
    # Refit on the FULL training set (never on validation or test data).
    final_pipeline = make_pipeline(model_name, BEST_STRATEGY[model_name]).set_params(**best_params)
    final_pipeline.fit(X_train, y_train)
    tuned_models[model_name] = final_pipeline
    tuning_rows.append({"model": model_name, "strategy": BEST_STRATEGY[model_name],
                        "cv_pr_auc": search.best_score_, "best_params": best_params,
                        "seconds": time.time() - started})
    print(f"{model_name:20s} CV PR-AUC={search.best_score_:.4f}  ({time.time() - started:.0f}s)  {best_params}")

display(pd.DataFrame(tuning_rows).round(4))

## 8. Validation-set comparison of the four tuned models

In [ ]:
def evaluate(y_true, probabilities, threshold=0.5):
    predicted = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    return {"threshold": threshold,
            "accuracy": accuracy_score(y_true, predicted),
            "precision": precision_score(y_true, predicted, zero_division=0),
            "recall": recall_score(y_true, predicted, zero_division=0),
            "f1": f1_score(y_true, predicted, zero_division=0),
            f"f{F_BETA:g}": fbeta_score(y_true, predicted, beta=F_BETA, zero_division=0),
            "roc_auc": roc_auc_score(y_true, probabilities),
            "pr_auc": average_precision_score(y_true, probabilities),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

THRESHOLD_GRID = np.round(np.arange(0.05, 0.951, 0.01), 2)

def threshold_sweep(y_true, probabilities):
    rows = []
    for threshold in THRESHOLD_GRID:
        predicted = (probabilities >= threshold).astype(int)
        rows.append({"threshold": threshold,
                     "precision": precision_score(y_true, predicted, zero_division=0),
                     "recall": recall_score(y_true, predicted, zero_division=0),
                     "f1": f1_score(y_true, predicted, zero_division=0),
                     "fbeta": fbeta_score(y_true, predicted, beta=F_BETA, zero_division=0)})
    return pd.DataFrame(rows)

validation_probabilities = {name: model.predict_proba(X_val)[:, 1] for name, model in tuned_models.items()}

comparison_rows = []
best_thresholds = {}
for name, probabilities in validation_probabilities.items():
    sweep = threshold_sweep(y_val, probabilities)
    best_thresholds[name] = float(sweep.loc[sweep["fbeta"].idxmax(), "threshold"])
    default_metrics = evaluate(y_val, probabilities, 0.5)
    tuned_metrics = evaluate(y_val, probabilities, best_thresholds[name])
    comparison_rows.append({
        "model": name, "strategy": BEST_STRATEGY[name],
        "ROC-AUC": default_metrics["roc_auc"], "PR-AUC": default_metrics["pr_auc"],
        "Acc@0.5": default_metrics["accuracy"], "Prec@0.5": default_metrics["precision"],
        "Rec@0.5": default_metrics["recall"], "F1@0.5": default_metrics["f1"],
        "best_threshold": best_thresholds[name],
        "Acc@best": tuned_metrics["accuracy"], "Prec@best": tuned_metrics["precision"],
        "Rec@best": tuned_metrics["recall"], "F1@best": tuned_metrics["f1"],
        f"F{F_BETA:g}@best": tuned_metrics[f"f{F_BETA:g}"],
    })

comparison = pd.DataFrame(comparison_rows).set_index("model").sort_values("PR-AUC", ascending=False)
display(comparison.round(4))
comparison.to_csv(WORKING_DIR / "model_comparison_validation.csv")

In [ ]:
# ROC and Precision-Recall curves (validation set)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, probabilities in validation_probabilities.items():
    fpr, tpr, _ = roc_curve(y_val, probabilities)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_val, probabilities):.3f})")
    precision_curve, recall_curve, _ = precision_recall_curve(y_val, probabilities)
    axes[1].plot(recall_curve, precision_curve,
                 label=f"{name} (AP={average_precision_score(y_val, probabilities):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[0].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC curves (validation)")
axes[1].axhline(y_val.mean(), color="k", linestyle="--", alpha=0.5, label=f"Baseline ({y_val.mean():.3f})")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision-Recall curves (validation)")
axes[0].legend(loc="lower right"); axes[1].legend(loc="upper right")
show(fig, "08_roc_pr_validation.png")

# Confusion matrices at each model's validation-selected threshold
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, name in zip(axes, comparison.index):
    threshold = best_thresholds[name]
    matrix = confusion_matrix(y_val, (validation_probabilities[name] >= threshold).astype(int))
    sns.heatmap(matrix, annot=True, fmt=",d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Pred 0", "Pred 1"], yticklabels=["True 0", "True 1"])
    ax.set_title(f"{name}\nthreshold={threshold:.2f}")
show(fig, "09_confusion_matrices_validation.png")

**Reading the errors in a screening context**
* **False positive (FP):** a person *without* the reported condition is flagged. Cost: unnecessary follow-up, worry.
* **False negative (FN):** a person *with* the reported condition is **missed**. In screening this is usually the costlier error, which is why the threshold is chosen with an **F2 score** (recall weighted 2× over precision) rather than F1 or accuracy.
* Lowering the threshold raises recall but lowers precision; the sweep below makes this trade-off explicit.

## 9. Model selection and threshold optimisation (validation data only)
**Selection rule:** highest validation **PR-AUC** (threshold-independent and robust under class imbalance), then sanity-checked against ROC-AUC, recall/precision and F2 at the chosen operating point. Accuracy is *not* used for selection because always predicting the majority class already scores high accuracy. The **test set has not been touched.**

In [ ]:
FINAL_MODEL_NAME = comparison["PR-AUC"].idxmax()
final_pipeline = tuned_models[FINAL_MODEL_NAME]
final_strategy = BEST_STRATEGY[FINAL_MODEL_NAME]
final_val_probabilities = validation_probabilities[FINAL_MODEL_NAME]

runner_up = comparison["PR-AUC"].sort_values(ascending=False).index[1]
print(f"Selected model: {FINAL_MODEL_NAME} (imbalance strategy: {final_strategy})")
print(f"Validation PR-AUC {comparison.loc[FINAL_MODEL_NAME, 'PR-AUC']:.4f} vs runner-up {runner_up} "
      f"{comparison.loc[runner_up, 'PR-AUC']:.4f}; ROC-AUC {comparison.loc[FINAL_MODEL_NAME, 'ROC-AUC']:.4f}.")

sweep = threshold_sweep(y_val, final_val_probabilities)
SELECTED_THRESHOLD = float(sweep.loc[sweep["fbeta"].idxmax(), "threshold"])
THRESHOLD_RULE = (f"Threshold maximising F{F_BETA:g} (recall weighted {F_BETA:g}x over precision) on the "
                  f"validation set, searched over {THRESHOLD_GRID[0]}-{THRESHOLD_GRID[-1]} in steps of 0.01.")
print(f"\nSelected threshold = {SELECTED_THRESHOLD:.2f}\n{THRESHOLD_RULE}")

display(sweep.iloc[::5].round(4).set_index("threshold"))
f1_best_threshold = float(sweep.loc[sweep["f1"].idxmax(), "threshold"])
print(f"For reference: F1-optimal threshold on validation = {f1_best_threshold:.2f}")

fig, ax = plt.subplots(figsize=(9, 5))
for column, label in [("precision", "Precision"), ("recall", "Recall"), ("f1", "F1"), ("fbeta", f"F{F_BETA:g}")]:
    ax.plot(sweep["threshold"], sweep[column], label=label)
ax.axvline(SELECTED_THRESHOLD, color="k", linestyle="--", label=f"Selected = {SELECTED_THRESHOLD:.2f}")
ax.axvline(0.5, color="gray", linestyle=":", label="Default 0.5")
ax.set(xlabel="Decision threshold", ylabel="Score", title=f"{FINAL_MODEL_NAME}: metrics vs threshold (validation)")
ax.legend()
show(fig, "10_threshold_sweep.png")

validation_metrics = evaluate(y_val, final_val_probabilities, SELECTED_THRESHOLD)
print("Validation metrics at selected threshold (optimistic: the threshold was tuned on this set):")
print({k: round(v, 4) if isinstance(v, float) else v for k, v in validation_metrics.items()})

## 10. Final evaluation on the untouched test set (performed once)
The final pipeline was fit on the **training** split only; the threshold was chosen on the **validation** split. The test probabilities are computed once and reused for every metric and plot below. No decision is made from these numbers.

In [ ]:
test_probabilities = final_pipeline.predict_proba(X_test)[:, 1]   # single pass over the test set
test_metrics = evaluate(y_test, test_probabilities, SELECTED_THRESHOLD)
test_metrics_default = evaluate(y_test, test_probabilities, 0.5)

test_table = pd.DataFrame({"selected threshold": test_metrics, "default 0.5 (reference)": test_metrics_default}).T
display(test_table[["threshold", "accuracy", "precision", "recall", "f1", f"f{F_BETA:g}", "roc_auc", "pr_auc",
                    "tn", "fp", "fn", "tp"]].round(4))

test_predictions = (test_probabilities >= SELECTED_THRESHOLD).astype(int)
print(f"Classification report on TEST set (threshold = {SELECTED_THRESHOLD:.2f}):")
print(classification_report(y_test, test_predictions, target_names=["No diabetes (0)", "Prediabetes/diabetes (1)"],
                            digits=4))

fig, axes = plt.subplots(1, 4, figsize=(21, 4.5))
for ax, (threshold, title) in zip(axes[:2], [(SELECTED_THRESHOLD, "Selected threshold"), (0.5, "Default 0.5")]):
    matrix = confusion_matrix(y_test, (test_probabilities >= threshold).astype(int))
    sns.heatmap(matrix, annot=True, fmt=",d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Pred 0", "Pred 1"], yticklabels=["True 0", "True 1"])
    ax.set_title(f"Test confusion matrix\n{title} ({threshold:.2f})")
fpr, tpr, _ = roc_curve(y_test, test_probabilities)
axes[2].plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, test_probabilities):.3f}")
axes[2].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[2].set(xlabel="False positive rate", ylabel="True positive rate", title="Test ROC curve")
axes[2].legend()
precision_curve, recall_curve, _ = precision_recall_curve(y_test, test_probabilities)
axes[3].plot(recall_curve, precision_curve, label=f"AP = {average_precision_score(y_test, test_probabilities):.3f}")
axes[3].axhline(y_test.mean(), color="k", linestyle="--", alpha=0.5, label="Baseline")
axes[3].scatter([test_metrics["recall"]], [test_metrics["precision"]], color="red", zorder=5,
                label="Selected threshold")
axes[3].set(xlabel="Recall", ylabel="Precision", title="Test Precision-Recall curve")
axes[3].legend()
show(fig, "11_test_evaluation.png")

In [ ]:
# Diagnostic only (no tuning): exact feature patterns can appear in both train and test because duplicates were
# kept. Check how much performance depends on such overlap.
train_patterns = set(map(tuple, X_train.to_numpy()))
test_in_train = np.array([tuple(row) in train_patterns for row in X_test.to_numpy()])
print(f"Test rows whose exact feature vector also appears in train: {test_in_train.sum():,} "
      f"({test_in_train.mean():.1%})")
if (~test_in_train).sum() > 0 and y_test[~test_in_train].nunique() == 2:
    unseen_metrics = evaluate(y_test[~test_in_train], test_probabilities[~test_in_train], SELECTED_THRESHOLD)
    print("Metrics on test rows NOT seen in train:",
          {k: round(unseen_metrics[k], 4) for k in ["precision", "recall", "f1", "roc_auc", "pr_auc"]})

## 11. Explainability
Three views of the **selected model**: built-in importance, permutation importance (model-agnostic, on validation data), and SHAP. These show what the model *relies on*, which reflects **statistical association in this survey, not causation** — e.g. a high SHAP value for `HighBP` does not mean blood pressure causes the label, and the model should not be used to infer what an individual should change.

In [ ]:
preprocessor = final_pipeline.named_steps["preprocess"]
final_model = final_pipeline.named_steps["model"]
transformed_feature_names = list(preprocessor.get_feature_names_out())

# 1) Built-in importance
if hasattr(final_model, "feature_importances_"):
    builtin_importance = pd.Series(final_model.feature_importances_, index=transformed_feature_names)
    builtin_label = "Built-in feature importance"
else:
    builtin_importance = pd.Series(np.abs(final_model.coef_[0]), index=transformed_feature_names)
    builtin_label = "|Coefficient| (standardised inputs)"

# 2) Permutation importance on a validation sample, scored with PR-AUC
sample_size = min(10_000, len(X_val))
X_val_sample = X_val.sample(sample_size, random_state=RANDOM_STATE)
y_val_sample = y_val.loc[X_val_sample.index]
permutation = permutation_importance(final_pipeline, X_val_sample, y_val_sample, scoring="average_precision",
                                     n_repeats=5, random_state=RANDOM_STATE, n_jobs=1)
permutation_series = pd.Series(permutation.importances_mean, index=X_val_sample.columns).sort_values()
permutation_std = pd.Series(permutation.importances_std, index=X_val_sample.columns)[permutation_series.index]

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
builtin_importance.sort_values().tail(15).plot.barh(ax=axes[0])
axes[0].set_title(builtin_label + " (top 15)")
permutation_series.tail(15).plot.barh(xerr=permutation_std.tail(15), ax=axes[1])
axes[1].set_title("Permutation importance: drop in PR-AUC (validation)")
show(fig, "12_feature_importance.png")

In [ ]:
# SHAP on a representative validation sample (stratified by the target)
shap_size = min(2_000, len(X_val))
shap_raw, _, _, _ = train_test_split(X_val, y_val, train_size=shap_size, stratify=y_val, random_state=RANDOM_STATE)
shap_frame = pd.DataFrame(preprocessor.transform(shap_raw), columns=transformed_feature_names, index=shap_raw.index)

if isinstance(final_model, LogisticRegression):
    background_raw = X_train.sample(min(1_000, len(X_train)), random_state=RANDOM_STATE)
    background = pd.DataFrame(preprocessor.transform(background_raw), columns=transformed_feature_names)
    explainer = shap.LinearExplainer(final_model, background)
    explanation = explainer(shap_frame)
else:
    explainer = shap.TreeExplainer(final_model)
    explanation = explainer(shap_frame, check_additivity=False)
if explanation.values.ndim == 3:       # some tree models return one slice per class
    explanation = explanation[:, :, 1]  # keep class 1 = prediabetes/diabetes

mean_abs_shap = pd.Series(np.abs(explanation.values).mean(axis=0), index=transformed_feature_names)
mean_abs_shap = mean_abs_shap.sort_values(ascending=False)
print("Top 10 features by mean |SHAP|:")
display(mean_abs_shap.head(10).round(4).to_frame("mean_abs_shap"))

plt.figure()
shap.plots.beeswarm(explanation, max_display=15, show=False)
plt.title("SHAP summary (beeswarm): impact on the model output for class 1")
show(plt.gcf(), "13_shap_summary.png")

plt.figure()
shap.plots.bar(explanation, max_display=15, show=False)
show(plt.gcf(), "14_shap_bar.png")

In [ ]:
# Feature-level explanation: dependence of SHAP values on the top features
top_features = list(mean_abs_shap.index[:4])
fig, axes = plt.subplots(1, len(top_features), figsize=(5 * len(top_features), 4))
for ax, feature in zip(np.atleast_1d(axes), top_features):
    shap.plots.scatter(explanation[:, feature], ax=ax, show=False)
    ax.set_title(f"SHAP dependence: {feature}")
show(fig, "15_shap_dependence.png")

# Individual-level explanation: highest and lowest scored examples in the SHAP sample
scores_in_sample = final_pipeline.predict_proba(shap_raw)[:, 1]
for label, position in [("highest score", int(np.argmax(scores_in_sample))),
                        ("lowest score", int(np.argmin(scores_in_sample)))]:
    plt.figure()
    shap.plots.waterfall(explanation[position], max_display=10, show=False)
    plt.title(f"Local explanation: {label} (model score {scores_in_sample[position]:.3f})")
    show(plt.gcf(), f"16_shap_waterfall_{label.replace(' ', '_')}.png")

**How to read the explanation (no causal claims).** Features with the largest mean |SHAP| are those the selected model uses most to separate the two groups *in this survey data*. In the beeswarm plot, red dots are high feature values and the horizontal position is the push toward (right) or away from (left) the "prediabetes/diabetes" output. Correlated features (e.g. general health, physical health days, difficulty walking, age, blood pressure) can share credit, and SHAP cannot separate correlation from cause. Use the printed ranking and plots above for the specific findings of this run.

## 12. Save the pipeline, threshold and the reusable prediction code
The complete preprocessing + model pipeline is saved with Joblib; the threshold and metadata go into a JSON file. `inference.py` (written below) holds the feature schema, input validation and the `predict_diabetes_risk` function, and is imported by both this notebook and the Streamlit app, so training and serving use the same feature names, order and pipeline.

In [ ]:
# ---- inference.py: schema, validation, loading and prediction (shared by notebook and Streamlit) ----
INFERENCE_CODE = r'''"""Inference utilities for the Diabetes Risk Screening Tool.

This module is the single source of truth for:
  * the feature schema (names, order, allowed values),
  * input validation,
  * loading the saved pipeline + threshold,
  * producing predictions.

The notebook and the Streamlit app both import it, so training-time and
serving-time feature handling cannot drift apart.

IMPORTANT: the model was trained on self-reported CDC BRFSS 2015 survey data.
It is a screening-style research/portfolio model, NOT a clinical diagnostic tool.
"""

from __future__ import annotations

import json
import warnings
from pathlib import Path
from typing import Any, Mapping

import joblib
import pandas as pd

PIPELINE_FILENAME = "diabetes_pipeline.joblib"
CONFIG_FILENAME = "model_config.json"
DEFAULT_ARTIFACT_DIR = Path(__file__).resolve().parent

YES_NO = {0: "No", 1: "Yes"}

AGE_GROUPS = {
    1: "18-24", 2: "25-29", 3: "30-34", 4: "35-39", 5: "40-44", 6: "45-49", 7: "50-54",
    8: "55-59", 9: "60-64", 10: "65-69", 11: "70-74", 12: "75-79", 13: "80 or older",
}
GENERAL_HEALTH = {1: "Excellent", 2: "Very good", 3: "Good", 4: "Fair", 5: "Poor"}
EDUCATION_LEVELS = {
    1: "Never attended school / kindergarten only",
    2: "Grades 1-8 (elementary)",
    3: "Grades 9-11 (some high school)",
    4: "Grade 12 or GED (high school graduate)",
    5: "College 1-3 years (some college / technical school)",
    6: "College 4+ years (college graduate)",
}
INCOME_LEVELS = {
    1: "Less than $10,000", 2: "$10,000 to < $15,000", 3: "$15,000 to < $20,000",
    4: "$20,000 to < $25,000", 5: "$25,000 to < $35,000", 6: "$35,000 to < $50,000",
    7: "$50,000 to < $75,000", 8: "$75,000 or more",
}


def _binary(label: str) -> dict[str, Any]:
    return {"kind": "binary", "label": label, "min": 0, "max": 1, "options": YES_NO}


# Order matches the column order of the original BRFSS 2015 CSV (target excluded).
FEATURE_SPECS: dict[str, dict[str, Any]] = {
    "HighBP": _binary("High blood pressure (ever told by a health professional)"),
    "HighChol": _binary("High cholesterol (ever told by a health professional)"),
    "CholCheck": _binary("Cholesterol check within the past 5 years"),
    "BMI": {"kind": "continuous", "label": "Body Mass Index (kg/m²)", "min": 10.0, "max": 100.0},
    "Smoker": _binary("Smoked at least 100 cigarettes in lifetime"),
    "Stroke": _binary("Ever told you had a stroke"),
    "HeartDiseaseorAttack": _binary("Coronary heart disease or myocardial infarction"),
    "PhysActivity": _binary("Physical activity in the past 30 days (outside of work)"),
    "Fruits": _binary("Eats fruit at least once per day"),
    "Veggies": _binary("Eats vegetables at least once per day"),
    "HvyAlcoholConsump": _binary("Heavy alcohol consumption"),
    "AnyHealthcare": _binary("Has any kind of health care coverage"),
    "NoDocbcCost": _binary("Could not see a doctor in the past 12 months because of cost"),
    "GenHlth": {"kind": "ordinal", "label": "General health", "min": 1, "max": 5, "options": GENERAL_HEALTH},
    "MentHlth": {"kind": "continuous", "label": "Days of poor mental health (past 30 days)", "min": 0, "max": 30},
    "PhysHlth": {"kind": "continuous", "label": "Days of poor physical health (past 30 days)", "min": 0, "max": 30},
    "DiffWalk": _binary("Serious difficulty walking or climbing stairs"),
    "Sex": {"kind": "binary", "label": "Sex", "min": 0, "max": 1, "options": {0: "Female", 1: "Male"}},
    "Age": {"kind": "ordinal", "label": "Age group", "min": 1, "max": 13, "options": AGE_GROUPS},
    "Education": {"kind": "ordinal", "label": "Education level", "min": 1, "max": 6, "options": EDUCATION_LEVELS},
    "Income": {"kind": "ordinal", "label": "Annual household income", "min": 1, "max": 8, "options": INCOME_LEVELS},
}
FEATURE_COLUMNS: list[str] = list(FEATURE_SPECS)


class InputValidationError(ValueError):
    """Raised when incoming data does not match the training schema."""


def validate_input(data: Mapping[str, Any] | list[Mapping[str, Any]] | pd.DataFrame) -> pd.DataFrame:
    """Validate raw input and return a DataFrame with exactly FEATURE_COLUMNS, in order.

    Accepts a single record (mapping), a list of records, or a DataFrame.
    Raises InputValidationError listing every problem found.
    """
    if isinstance(data, pd.DataFrame):
        frame = data.copy()
    elif isinstance(data, Mapping):
        frame = pd.DataFrame([dict(data)])
    elif isinstance(data, list) and data and all(isinstance(item, Mapping) for item in data):
        frame = pd.DataFrame([dict(item) for item in data])
    else:
        raise InputValidationError("Input must be a mapping, a non-empty list of mappings, or a DataFrame.")

    if frame.empty:
        raise InputValidationError("Input contains no rows.")

    errors: list[str] = []
    missing = [col for col in FEATURE_COLUMNS if col not in frame.columns]
    unexpected = [col for col in frame.columns if col not in FEATURE_COLUMNS]
    if missing:
        errors.append(f"Missing features: {missing}")
    if unexpected:
        errors.append(f"Unexpected features: {unexpected}")
    if errors:
        raise InputValidationError("; ".join(errors))

    frame = frame[FEATURE_COLUMNS]
    for column, spec in FEATURE_SPECS.items():
        numeric = pd.to_numeric(frame[column], errors="coerce")
        if numeric.isna().any():
            errors.append(f"'{column}' must be numeric and not missing.")
            continue
        if ((numeric < spec["min"]) | (numeric > spec["max"])).any():
            errors.append(f"'{column}' must be between {spec['min']} and {spec['max']}.")
            continue
        if spec["kind"] in ("binary", "ordinal") and (numeric % 1 != 0).any():
            errors.append(f"'{column}' must be a whole number.")
            continue
        frame[column] = numeric.astype("float64")

    if errors:
        raise InputValidationError("; ".join(errors))
    return frame


def load_artifacts(artifact_dir: str | Path | None = None):
    """Load the saved pipeline and its configuration (threshold, metrics, versions)."""
    directory = Path(artifact_dir) if artifact_dir else DEFAULT_ARTIFACT_DIR
    pipeline_path = directory / PIPELINE_FILENAME
    config_path = directory / CONFIG_FILENAME
    for path in (pipeline_path, config_path):
        if not path.is_file():
            raise FileNotFoundError(
                f"Required artifact not found: {path}. Run the notebook first and place "
                f"'{PIPELINE_FILENAME}' and '{CONFIG_FILENAME}' in {directory}."
            )

    config = json.loads(config_path.read_text(encoding="utf-8"))
    if list(config["feature_columns"]) != FEATURE_COLUMNS:
        raise RuntimeError("Feature columns in model_config.json do not match inference.py.")

    # Pickled models can misbehave when library versions differ, so warn loudly.
    import sklearn

    trained_with = config.get("library_versions", {}).get("scikit-learn")
    if trained_with and trained_with != sklearn.__version__:
        warnings.warn(
            f"Model trained with scikit-learn {trained_with}, but {sklearn.__version__} is installed. "
            "Install the versions in requirements.txt to avoid inconsistent predictions."
        )

    pipeline = joblib.load(pipeline_path)
    return pipeline, config


def predict_diabetes_risk(
    data: Mapping[str, Any] | list[Mapping[str, Any]] | pd.DataFrame,
    pipeline=None,
    config: dict[str, Any] | None = None,
    artifact_dir: str | Path | None = None,
):
    """Validate input, score it, and apply the saved decision threshold.

    Returns a dict for a single mapping, otherwise a list of dicts, each containing:
      predicted_class : 1 if score >= threshold else 0
      predicted_label : human-readable (non-diagnostic) wording
      probability     : model score for class 1 (see note below)
      threshold       : decision threshold selected on validation data

    Note: when class weights or SMOTE are used, the score is NOT a calibrated
    probability of having diabetes. Treat it as a ranking/screening score.
    """
    if pipeline is None or config is None:
        pipeline, config = load_artifacts(artifact_dir)

    frame = validate_input(data)
    scores = pipeline.predict_proba(frame)[:, 1]
    threshold = float(config["threshold"])

    results = []
    for score in scores:
        predicted_class = int(score >= threshold)
        results.append({
            "predicted_class": predicted_class,
            "predicted_label": (
                "Pattern associated with self-reported prediabetes/diabetes"
                if predicted_class == 1
                else "Pattern not associated with self-reported prediabetes/diabetes"
            ),
            "probability": float(score),
            "threshold": threshold,
        })
    return results[0] if isinstance(data, Mapping) else results
'''
(WORKING_DIR / "inference.py").write_text(INFERENCE_CODE, encoding="utf-8")
print("Wrote", WORKING_DIR / "inference.py")

In [ ]:
# ---- app.py: Streamlit application ----
APP_CODE = r'''"""Streamlit app: Diabetes Risk Screening Tool (educational portfolio project).

Run with:  streamlit run app.py
Requires in the same folder: inference.py, diabetes_pipeline.joblib, model_config.json
(Optionally set DIABETES_ARTIFACT_DIR to load the artifacts from another folder.)
"""

import os

import streamlit as st

from inference import (
    FEATURE_SPECS,
    InputValidationError,
    load_artifacts,
    predict_diabetes_risk,
)

DISCLAIMER = (
    "**Medical disclaimer:** This tool is an educational machine-learning demonstration trained on "
    "self-reported 2015 CDC BRFSS survey data. It is **not** a medical device, has **not** been "
    "clinically validated, and does **not** diagnose diabetes or predict future diabetes onset. "
    "Its output must not be used to make health decisions. Please consult a qualified health "
    "professional for diagnosis, testing, or advice."
)

st.set_page_config(page_title="Diabetes Risk Screening Tool", page_icon="🩺", layout="wide")


@st.cache_resource(show_spinner="Loading model...")
def get_artifacts():
    return load_artifacts(os.environ.get("DIABETES_ARTIFACT_DIR"))


def render_selector(name: str, key_prefix: str = "") -> float:
    """Render the right widget for a feature and return the chosen numeric value."""
    spec = FEATURE_SPECS[name]
    key = f"{key_prefix}{name}"
    if spec["kind"] == "binary" and set(spec["options"]) == {0, 1}:
        return st.radio(
            spec["label"], list(spec["options"]), format_func=spec["options"].get,
            horizontal=True, key=key,
        )
    if spec["kind"] in ("binary", "ordinal"):
        options = list(spec["options"])
        default_index = 0 if spec["kind"] == "binary" else len(options) // 2
        return st.selectbox(
            spec["label"], options, index=default_index,
            format_func=spec["options"].get, key=key,
        )
    return st.number_input(
        spec["label"], min_value=int(spec["min"]), max_value=int(spec["max"]),
        value=0, step=1, key=key,
    )


def main() -> None:
    st.title("🩺 Diabetes Risk Screening Tool")
    st.warning(DISCLAIMER)

    try:
        pipeline, config = get_artifacts()
    except (FileNotFoundError, RuntimeError) as error:
        st.error(str(error))
        st.stop()

    with st.sidebar:
        st.header("About the model")
        st.write(f"**Model:** {config['model_name']}")
        st.write(f"**Imbalance strategy:** {config['imbalance_strategy']}")
        st.write(f"**Decision threshold:** {config['threshold']:.2f}")
        st.caption(config["threshold_rule"])
        test_metrics = config.get("test_metrics", {})
        if test_metrics:
            st.subheader("Held-out test performance")
            for label, key in [("Recall", "recall"), ("Precision", "precision"),
                               ("F1", "f1"), ("ROC-AUC", "roc_auc"), ("PR-AUC", "pr_auc")]:
                if key in test_metrics:
                    st.write(f"{label}: {test_metrics[key]:.3f}")
        st.caption(
            "Target = self-reported prediabetes OR diabetes (BRFSS binary file). "
            "Scores are not calibrated probabilities."
        )

    st.subheader("Enter health indicators")
    with st.form("screening_form"):
        columns = st.columns(3)

        with columns[0]:
            st.markdown("**Body measurements & demographics**")
            height_cm = st.number_input("Height (cm)", min_value=100, max_value=230, value=170, step=1)
            weight_kg = st.number_input("Weight (kg)", min_value=25, max_value=300, value=75, step=1)
            bmi = round(weight_kg / ((height_cm / 100) ** 2), 1)
            st.caption(f"Calculated BMI: **{bmi}**")
            values = {"BMI": bmi}
            for name in ("Sex", "Age", "Education", "Income"):
                values[name] = render_selector(name)

        with columns[1]:
            st.markdown("**Medical history**")
            for name in ("HighBP", "HighChol", "CholCheck", "Stroke", "HeartDiseaseorAttack", "DiffWalk"):
                values[name] = render_selector(name)

        with columns[2]:
            st.markdown("**Lifestyle & health status**")
            for name in ("Smoker", "PhysActivity", "Fruits", "Veggies", "HvyAlcoholConsump",
                         "AnyHealthcare", "NoDocbcCost", "GenHlth", "MentHlth", "PhysHlth"):
                values[name] = render_selector(name)

        submitted = st.form_submit_button("Run screening", type="primary")

    if not submitted:
        return

    try:
        result = predict_diabetes_risk(values, pipeline=pipeline, config=config)
    except InputValidationError as error:
        st.error(f"Invalid input: {error}")
        return

    st.subheader("Result")
    score = result["probability"]
    left, right = st.columns(2)
    left.metric("Model score", f"{score:.1%}")
    right.metric("Decision threshold", f"{result['threshold']:.0%}")
    st.progress(min(max(score, 0.0), 1.0))

    if result["predicted_class"] == 1:
        st.warning(f"**{result['predicted_label']}.** The score is at or above the screening threshold.")
    else:
        st.success(f"**{result['predicted_label']}.** The score is below the screening threshold.")

    st.info(
        "This is **not a diagnosis**. The score reflects statistical patterns in self-reported survey data "
        "from 2015 and is not a calibrated probability of having diabetes. A low score does not rule out "
        "diabetes, and a high score does not confirm it. Please speak with a health professional."
    )
    st.caption(DISCLAIMER)


main()
'''
(WORKING_DIR / "app.py").write_text(APP_CODE, encoding="utf-8")
import py_compile
py_compile.compile(str(WORKING_DIR / "app.py"), doraise=True)  # syntax check
print("Wrote and syntax-checked", WORKING_DIR / "app.py")

In [ ]:
from importlib import metadata

def installed_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None

LIBRARIES = ["scikit-learn", "imbalanced-learn", "xgboost", "joblib", "pandas", "numpy", "shap", "streamlit"]
library_versions = {name: installed_version(name) for name in LIBRARIES if installed_version(name)}

def plain(values):
    return {k: (float(v) if isinstance(v, (float, np.floating)) else int(v) if isinstance(v, (int, np.integer)) else v)
            for k, v in values.items()}

PIPELINE_PATH = WORKING_DIR / "diabetes_pipeline.joblib"
CONFIG_PATH = WORKING_DIR / "model_config.json"

joblib.dump(final_pipeline, PIPELINE_PATH)

model_config = {
    "project": "Diabetes Risk Screening Tool",
    "model_name": FINAL_MODEL_NAME,
    "imbalance_strategy": final_strategy,
    "target": TARGET,
    "target_meaning": "0 = no diabetes, 1 = prediabetes or diabetes (self-reported, BRFSS 2015)",
    "threshold": SELECTED_THRESHOLD,
    "threshold_rule": THRESHOLD_RULE,
    "feature_columns": FEATURES,
    "validation_metrics": plain(validation_metrics),
    "test_metrics": plain(test_metrics),
    "random_state": RANDOM_STATE,
    "library_versions": library_versions,
    "disclaimer": ("Educational model trained on self-reported survey data. Not clinically validated; "
                   "not a diagnostic tool; does not predict future onset; scores are not calibrated probabilities."),
}
CONFIG_PATH.write_text(json.dumps(model_config, indent=2), encoding="utf-8")

requirements = ["streamlit", "scikit-learn", "imbalanced-learn", "xgboost", "joblib", "pandas", "numpy"]
requirement_lines = [f"{name}=={library_versions[name]}" if name in library_versions else name for name in requirements]
(WORKING_DIR / "requirements.txt").write_text("\n".join(requirement_lines) + "\n", encoding="utf-8")

print("Saved:", PIPELINE_PATH, CONFIG_PATH, WORKING_DIR / "requirements.txt", sep="\n  ")
print("\nrequirements.txt:\n" + "\n".join(requirement_lines))

## 13. Verify that inference matches training (consistency & validation tests)

In [ ]:
sys.path.insert(0, str(WORKING_DIR))
import importlib
import inference
importlib.reload(inference)

assert inference.FEATURE_COLUMNS == FEATURES, "Schema in inference.py differs from the training feature order"

loaded_pipeline, loaded_config = inference.load_artifacts(WORKING_DIR)

# 1) Reloaded pipeline + inference function reproduce the in-memory pipeline exactly.
check_rows = X_test.sample(min(300, len(X_test)), random_state=RANDOM_STATE)
direct_scores = final_pipeline.predict_proba(check_rows)[:, 1]
api_results = inference.predict_diabetes_risk(check_rows, pipeline=loaded_pipeline, config=loaded_config)
api_scores = np.array([r["probability"] for r in api_results])
api_classes = np.array([r["predicted_class"] for r in api_results])
assert np.allclose(direct_scores, api_scores), "Saved pipeline output differs from the trained pipeline"
assert np.array_equal(api_classes, (direct_scores >= SELECTED_THRESHOLD).astype(int))

# 2) Column order in the incoming data must not matter.
shuffled = check_rows[list(reversed(check_rows.columns))]
shuffled_scores = np.array([r["probability"] for r in inference.predict_diabetes_risk(shuffled, loaded_pipeline, loaded_config)])
assert np.allclose(direct_scores, shuffled_scores)
print("Consistency checks passed (reloaded pipeline == trained pipeline; column order irrelevant).")

# 3) Input validation rejects bad data.
valid_record = check_rows.iloc[0].to_dict()
bad_inputs = {
    "missing feature": {k: v for k, v in valid_record.items() if k != "BMI"},
    "unexpected feature": {**valid_record, "Glucose": 120},
    "out-of-range BMI": {**valid_record, "BMI": 500},
    "non-binary flag": {**valid_record, "HighBP": 2},
    "non-integer ordinal": {**valid_record, "Age": 3.5},
    "non-numeric value": {**valid_record, "BMI": "abc"},
}
for description, record in bad_inputs.items():
    try:
        inference.predict_diabetes_risk(record, loaded_pipeline, loaded_config)
    except inference.InputValidationError as error:
        print(f"Rejected ({description}): {error}")
    else:
        raise AssertionError(f"Invalid input was accepted: {description}")

# 4) Example prediction
print("\nExample prediction for one test record:")
print(inference.predict_diabetes_risk(valid_record, loaded_pipeline, loaded_config), "| true label:", int(y_test.loc[check_rows.index[0]]))

## 14. Final review checklist
| Check | How it is handled in this notebook |
|---|---|
| Target interpretation | `Diabetes_binary`: 0 = no diabetes, 1 = prediabetes/diabetes; target excluded from `FEATURES` (asserted) |
| No data leakage | Split *before* any fitting; imputers/scalers/SMOTE/model live inside pipelines fit on training data (or training CV folds) only |
| Imbalance handling | Baseline vs class weights vs SMOTE compared by CV; SMOTE only inside the training fold/set; validation & test left unchanged |
| Threshold | Chosen on the **validation** set (F2); test set never used for any decision |
| Test set | Probabilities computed once, after all choices were frozen |
| Duplicates / zeros | Investigated and kept with justification; overlap diagnostic reported |
| Train/serve consistency | One saved pipeline; shared `inference.py` schema + validation; reload & column-order tests |
| Paths | Input located programmatically; everything written to `/kaggle/working/` |
| Dependencies | Missing packages installed at the top; `requirements.txt` pins the versions used |
| Limitations | Stated at the top, in the app sidebar/disclaimer and in the saved config |

## 15. Using the model with the Streamlit app
Files produced in `/kaggle/working/` (download them from the notebook's **Output** tab): `diabetes_pipeline.joblib`, `model_config.json`, `inference.py`, `app.py`, `requirements.txt`.

1. Put those five files together in one folder on your computer.
2. `pip install -r requirements.txt` (pins the library versions used for training; the pickled pipeline needs them).
3. `streamlit run app.py`
4. Programmatic use: `from inference import predict_diabetes_risk` then `predict_diabetes_risk({...21 features...})` returns `predicted_class`, `probability`, `threshold`.